In [2]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics.pairwise import cosine_similarity
import gc
from sklearn.feature_extraction.text import TfidfVectorizer
import itertools
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords', quiet=True)

True

Step 1: Load the Data

In [3]:
# Load the datasets
interactions = pd.read_csv('https://raw.githubusercontent.com/ywh1021/MA2_ML/refs/heads/main/interactions_train.csv')
books = pd.read_csv("https://raw.githubusercontent.com/ywh1021/MA2_ML/refs/heads/main/items.csv", encoding='utf-8')
submission_sample = pd.read_csv("https://raw.githubusercontent.com/ywh1021/MA2_ML/refs/heads/main/sample_submission.csv")

# Display the first rows of each dataset
display(interactions.head())
display(books.head())

,u,i,t
0,4456,8581,1.687541e+09
1,142,1964,1.679585e+09
2,362,3705,1.706872e+09
3,1809,11317,1.673533e+09
4,4384,1323,1.681402e+09


,Title,Author,ISBN Valid,Publisher,Subjects,i
0,Classification décimale universelle : édition ...,NaN,9782871303336; 2871303339,Ed du CEFAL,Classification décimale universelle; Indexatio...,0
1,Les interactions dans l'enseignement des langu...,"Cicurel, Francine, 1947-",9782278058327; 2278058320,Didier,didactique--langue étrangère - enseignement; d...,1
2,Histoire de vie et recherche biographique : pe...,NaN,2343190194; 9782343190198,L'Harmattan,Histoires de vie en sociologie; Sciences socia...,2
3,Ce livre devrait me permettre de résoudre le c...,"Mazas, Sylvain, 1980-",9782365350020; 236535002X; 9782365350488; 2365...,Vraoum!,Moyen-Orient; Bandes dessinées autobiographiqu...,3
4,Les années glorieuses : roman /,"Lemaitre, Pierre, 1951-",9782702180815; 2702180817; 9782702183618; 2702...,Calmann-Lévy,France--1945-1975; Roman historique; Roman fra...,4


Check the Number of interactions, users and books

In [4]:
# check number of interaction, users, books
n_users = interactions.u.nunique()
n_items = books.i.nunique()
print('number of users =', n_users, '| number of books =', n_items)

number of users = 7838 | number of books = 15291


In [6]:
# ==========================================
# TF-IDF Content-Based Feature Engineering
# ==========================================
print("\nProcessing Content Features (TF-IDF)...")

# 1. Handle the missing values identified in the EDA
for col in ['Title', 'Author', 'Publisher', 'Subjects']:
    books[col] = books[col].fillna('')

# 2. Combine metadata into a single string per book
books['combined_features'] = books['Title'] + ' ' + books['Author'] + ' ' + books['Publisher'] + ' ' + books['Subjects']

# use french stop words
french_stop_words = stopwords.words('french')

# 3. Vectorize the text data (Limit features to prevent memory overload)
tfidf = TfidfVectorizer(stop_words=french_stop_words, max_features=30000)
tfidf_matrix = tfidf.fit_transform(books['combined_features'])

# 4. Calculate Item-Item cosine similarity based on content
content_sim_global = cosine_similarity(tfidf_matrix)

# Free up memory
del tfidf_matrix
gc.collect()
print("TF-IDF matrix generated successfully.")


Processing Content Features (TF-IDF)...
TF-IDF matrix generated successfully.


In [7]:
# sort the interactions by user and time stamp
interactions = interactions.sort_values(["u", "t"])
interactions["pct_rank"] = interactions.groupby("u")["t"].rank(pct=True, method='dense')
interactions.reset_index(inplace=True, drop=True)
display(interactions)

,u,i,t,pct_rank
0,0,0,1.680191e+09,0.040000
1,0,1,1.680783e+09,0.080000
2,0,2,1.680801e+09,0.120000
3,0,3,1.683715e+09,0.160000
4,0,3,1.683715e+09,0.200000
...,...,...,...,...
87042,7836,3471,1.728644e+09,0.666667
87043,7836,3471,1.728644e+09,1.000000
87044,7837,2191,1.728735e+09,0.333333
87045,7837,88,1.728735e+09,0.666667


In [8]:
# Define a function to create the data matrix
def create_data_matrix(data, n_users, n_items):
    data_matrix = np.zeros((n_users, n_items))
    data_matrix[data["u"].values, data["i"].values] = 1

    return data_matrix


# Define the function to predict interactions based on item similarity
def item_based_predict(interactions, similarity, epsilon=1e-9):
    pred = similarity.dot(interactions.T) / (similarity.sum(axis=1)[:, np.newaxis] + epsilon) # epsilon is for avoiding dividing 0
    return pred.T  # Transpose to get users as rows and items as columns


# Define the function to predict interactions based on user similarity
def user_based_predict(interactions, similarity, epsilon=1e-9):
    pred = similarity.dot(interactions) / (np.abs(similarity).sum(axis=1)[:, np.newaxis] + epsilon)
    return pred


# Implement the precision_recall_at_k function
def precision_recall_at_k(prediction, ground_truth, k=10):
    num_users = prediction.shape[0]
    precision_at_k, recall_at_k = 0, 0

    for user in range(num_users):
        top_k_items = np.argpartition(prediction[user], -k)[-k:]
        relevant_items_in_top_k = np.sum(ground_truth[user][top_k_items])
        total_relevant_items = np.sum(ground_truth[user])

        # Update Precision@K and Recall@K for this user
        precision_at_k += relevant_items_in_top_k / k
        recall_at_k += relevant_items_in_top_k / total_relevant_items if total_relevant_items > 0 else 0

    # Calculate the average Precision@K and Recall@K over all users
    precision_at_k /= num_users
    recall_at_k /= num_users

    return precision_at_k, recall_at_k

In [9]:
# ==========================================
# 3-Way Grid Search Weights Initialization
# ==========================================

k_folds = 5

# Generate all weight combinations (User, Item, Content) that sum to 1.0
# Using 0.1 (10%) steps creates 66 combinations, which is optimal for CV performance
weight_combinations = []
steps = np.linspace(0, 1, 11)
for wu in steps:
    for wi in steps:
        wc = 1.0 - wu - wi
        if -0.001 <= wc <= 1.001: # Handle floating point precision
            weight_combinations.append((np.round(wu, 2), np.round(wi, 2), np.round(wc, 2)))

# Use dictionary to store CV results for tuples of weights
hybrid_precision_cv = {w: [] for w in weight_combinations}
hybrid_recall_cv = {w: [] for w in weight_combinations}

print(f"\nStarting {k_folds}-Fold Cross Validation with 3-Way Hybrid Search...")
print(f"Testing {len(weight_combinations)} weight combinations...")

for fold in range(k_folds):
    test_lower = fold / k_folds
    test_upper = (fold + 1) / k_folds

    if fold == k_folds - 1:
        test_mask = (interactions["pct_rank"] >= test_lower) & (interactions["pct_rank"] <= test_upper)
    else:
        test_mask = (interactions["pct_rank"] >= test_lower) & (interactions["pct_rank"] < test_upper)

    train_mask = ~test_mask

    train_data_cv = interactions[train_mask]
    test_data_cv = interactions[test_mask]

    # built Data Matrices
    train_matrix_cv = create_data_matrix(train_data_cv, n_users, n_items)
    test_matrix_cv = create_data_matrix(test_data_cv, n_users, n_items)

    # -------------------------------------------
    # Stage 1: generate Item-Based prediction matrix
    # -------------------------------------------
    item_sim_cv = cosine_similarity(train_matrix_cv.T)
    item_pred_cv = item_based_predict(train_matrix_cv, item_sim_cv)

    # delete matrix to recover memory
    del item_sim_cv
    gc.collect()

    # -------------------------------------------
    # Stage 2: generate User-Based prediction matrix
    # -------------------------------------------
    user_sim_cv = cosine_similarity(train_matrix_cv)
    user_pred_cv = user_based_predict(train_matrix_cv, user_sim_cv)

    # delete matrix to recover memory
    del user_sim_cv
    gc.collect()

    # -------------------------------------------
    # Stage 3: generate Content-Based prediction matrix
    # -------------------------------------------
    # We use the global content similarity matrix with the training interactions
    content_pred_cv = item_based_predict(train_matrix_cv, content_sim_global)


    # -------------------------------------------
    # Stage 4: 3-Way Grid Search
    # -------------------------------------------
    for w in weight_combinations:
        wu, wi, wc = w

        # wu * User + wi * Item + wc * Content
        hybrid_pred = (wu * user_pred_cv) + (wi * item_pred_cv) + (wc * content_pred_cv)

        # evaluate and record
        prec, rec = precision_recall_at_k(hybrid_pred, test_matrix_cv, k=10)
        hybrid_precision_cv[w].append(prec)
        hybrid_recall_cv[w].append(rec)

        # delete matrix for next combination
        del hybrid_pred

    # delete prediction matrices after every fold
    del item_pred_cv, user_pred_cv, content_pred_cv
    gc.collect()

    print(f"Fold {fold + 1} completed | Train size: {len(train_data_cv)}, Test size: {len(test_data_cv)}")

# ==========================================
# Grid Search redult and best weight
# ==========================================
print("\n=== 3-Way Hybrid CF Grid Search Results ===")
best_weights = None
best_prec = -1
best_rec = -1
best_prec_std = 0
best_rec_std = 0

for w in weight_combinations:
    wu, wi, wc = w

    # calculate mean and std for 5-fold
    avg_prec = np.mean(hybrid_precision_cv[w])
    std_prec = np.std(hybrid_precision_cv[w])
    avg_rec = np.mean(hybrid_recall_cv[w])
    std_rec = np.std(hybrid_recall_cv[w])

    # Only print combinations that perform reasonably well to keep output clean,
    # or you can comment out the print statement entirely if it's too noisy.
    if avg_prec > 0:
        pass # print(f"Weight [User {wu*100:3.0f}% | Item {wi*100:3.0f}% | Content {wc*100:3.0f}%] -> Precision@10: {avg_prec:.4f}")

    # use Precision@10 to find best weight
    if avg_prec > best_prec:
        best_prec = avg_prec
        best_rec = avg_rec
        best_prec_std = std_prec
        best_rec_std = std_rec
        best_weights = w

print("-" * 50)
wu_best, wi_best, wc_best = best_weights
print(f"Best Combination : User {wu_best*100:.0f}% | Item {wi_best*100:.0f}% | Content {wc_best*100:.0f}%")
print(f"Average Precision@10: {best_prec:.4f} ± {best_prec_std:.4f}")
print(f"Average Recall@10:    {best_rec:.4f} ± {best_rec_std:.4f}")


Starting 5-Fold Cross Validation with 3-Way Hybrid Search...
Testing 66 weight combinations...
Fold 1 completed | Train size: 74423, Test size: 12624
Fold 2 completed | Train size: 68779, Test size: 18268
Fold 3 completed | Train size: 70789, Test size: 16258
Fold 4 completed | Train size: 68778, Test size: 18269
Fold 5 completed | Train size: 65419, Test size: 21628

=== 3-Way Hybrid CF Grid Search Results ===
--------------------------------------------------
Best Combination : User 20% | Item 10% | Content 70%
Average Precision@10: 0.0532 ± 0.0099
Average Recall@10:    0.2741 ± 0.0694


In [10]:
# filter top 10 prediction
def get_top_10_df(prediction, k=10):
    num_users = prediction.shape[0]
    recommendation_list = []

    for user in range(num_users):
        # Step 1: get top 10 index
        top_k_items = np.argsort(prediction[user])[-k:][::-1]

        # Step 2: use " " to separate strings
        rec_str = " ".join(top_k_items.astype(str))

        # Step 3: save user_id and recommendation strings
        recommendation_list.append({
            "user_id": user,
            "recommendation": rec_str
        })

    # Step 4: transform to dataframe
    df = pd.DataFrame(recommendation_list)
    return df

## User-Item Hybrid Collaborative Filtering (All interaction data used)

In [11]:
# ==========================================
# Final Hybrid Prediction (100% Data)
# ==========================================
print("=== Generating Final Hybrid Prediction Matrix ===")

# best weight
w_user = 0.20
w_item = 0.10
w_content = 0.70

# 1. 100% full data matrix
print("1. Creating full data matrix...")
train_data_matrix_all = create_data_matrix(interactions, n_users, n_items)

# 2. Item prediction
print("2. Computing Item-based predictions...")
item_similarity_all = cosine_similarity(train_data_matrix_all.T)
item_prediction_all = item_based_predict(train_data_matrix_all, item_similarity_all)

# delete Item similarity matrix
del item_similarity_all
gc.collect()

# 3. User prediction
print("3. Computing User-based predictions...")
user_similarity_all = cosine_similarity(train_data_matrix_all)
user_prediction_all = user_based_predict(train_data_matrix_all, user_similarity_all)

# delete Item similarity matrix
del user_similarity_all
gc.collect()

# 4. Content prediction
print("4. Computing Content-based predictions...")
content_prediction_all = item_based_predict(train_data_matrix_all, content_sim_global)


# 5. Hybrid Prediction Matrix
print(f"5. Fusing into Hybrid Prediction Matrix (User {w_user*100:.0f}% | Item {w_item*100:.0f}% | Content {w_content*100:.0f}%)...")
hybrid_prediction_all = (w_user * user_prediction_all) + \
                        (w_item * item_prediction_all) + \
                        (w_content * content_prediction_all)

# release RAM
del user_prediction_all, item_prediction_all, content_prediction_all
gc.collect()

print("✓ Final 3-Way Hybrid Prediction Matrix is ready! Shape:", hybrid_prediction_all.shape)

=== Generating Final Hybrid Prediction Matrix ===
1. Creating full data matrix...
2. Computing Item-based predictions...
3. Computing User-based predictions...
4. Computing Content-based predictions...
5. Fusing into Hybrid Prediction Matrix (User 20% | Item 10% | Content 70%)...
✓ Final 3-Way Hybrid Prediction Matrix is ready! Shape: (7838, 15291)


In [12]:
top_10_user_df_all = get_top_10_df(hybrid_prediction_all, k=10)

# 2. preview the results
print(top_10_user_df_all.head())

# 3. csv
# top_10_user_df_all.to_csv("recommendations_user_alldata.csv", index=False)

   user_id                           recommendation
0        0                1 24 20 21 0 17 2 7 18 13
1        1            31 39 30 38 33 32 36 29 34 35
2        2            85 80 76 92 94 91 79 52 78 59
3        3  121 123 127 157 138 155 151 132 140 145
4        4  192 203 205 207 195 202 204 191 200 197
